In [ ]:
import pandas as pd
# 1. Importation des données
df= pd.read_csv("store.csv")

In [ ]:
# 2. Identification des anomalies avec l'IQR

Q1, Q3 = df['Sales'].quantile([0.25, 0.75])
limit = Q3 + 1.5 * (Q3 - Q1)
anos = df[df['Sales'] > limit]
print(f"Nombre d'anomalies (> {limit:.2f} $) : {len(anos)}")
print("\nLes Catégories qui ont le plus d'anomalies :\n", anos['Category'].value_counts().head(3))

In [ ]:
# 3. Nettoyage des données

df.dropna(subset=['Postal Code'], inplace=True)
df.drop_duplicates(inplace=True)
print(df.columns)
df.info()

In [ ]:
# 4. Formatage des dates
df['Order Date']= pd.to_datetime(df['Order Date'],format="%d/%m/%Y",errors='coerce')
df['Ship Date']= pd.to_datetime(df['Ship Date'],format="%d/%m/%Y",errors='coerce')
# 5. Extraction temporelle
df['years']=df['Order Date'].dt.year
df['quarter']=df['Order Date'].dt.quarter
df['months']=df['Order Date'].dt.month
# 6. Calculs financiers (Finance)
df['cost']=(df['Sales']*0.60).round(2)
df['profit']=(df['Sales']-df['cost']).round(2)
df['ratio_profit']=((df['profit'] / df['Sales'])*100).round(2)

In [ ]:
# 7. Calcul du délai de livraison (Logistique)
df['delivery_days']=(df['Ship Date']-df['Order Date']).dt.days

In [ ]:
# 8. Segmentation des clients (Commercial)
def segment_valeur(valeur):
    if valeur > 500:
        return 'high value'
    elif valeur > 250:
        return 'medium value'
    else: 
        return 'low value'
# 9. Agrégation des données (KPIs)
df['categorise']=df['Sales'].apply(segment_valeur)

In [ ]:
rev_p_rj=df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
rev_p_prd=df.groupby(["Product ID","Product Name"])["Sales"].sum().sort_values(ascending=False)
rev_p_prd
rev_p_rj

In [ ]:
# 10. Exportation finale
df.to_csv("store_new.csv",index=False)